# DIRTY ENERGY GETS AWAY WITH MORE AND MORE AS WORLD BURNS

Have you ever wondered how much and what kind of damage dirty energy does? Have you wondered how much dirty energy gets away with, and who bears the costs? And is the world heading in the right direction? Here, you can see the historical data (2015-2025) and projections (2026-2030).

Setup and data come from https://www.kaggle.com/code/zkskhurram/imf-global-fossil-fuel-subsidies-2015-2030/notebook

In [1]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import json
import os

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# ---- Matplotlib / Seaborn Style ----
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'figure.dpi': 100,
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'font.family': 'sans-serif',
})

# ---- Plotly Template ----
PLOTLY_TEMPLATE = 'plotly_white'
COLOR_PALETTE = px.colors.qualitative.Bold

# ---- Pandas Display ----
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', 60)

print('✅ All libraries loaded successfully!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   plotly  : {px.__version__ if hasattr(px, "__version__") else "5.x"}')

✅ All libraries loaded successfully!
   pandas  : 2.2.2
   numpy   : 2.0.2
   plotly  : 5.x


In [2]:
# Load the datasets
df_usd = pd.read_parquet('/content/imf_ffs_usd_cleaned.parquet')
df_gdp = pd.read_parquet('/content/imf_ffs_gdp_cleaned.parquet')

print("--- USD Dataset Info ---")
print(df_usd.info())

print("\n--- GDP Dataset Info ---")
print(df_gdp.info())

# Display the first few rows of the USD dataset
df_usd.head()

--- USD Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56448 entries, 0 to 56447
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_code    56448 non-null  object 
 1   country         56448 non-null  object 
 2   indicator_code  56448 non-null  object 
 3   indicator       56448 non-null  object 
 4   unit_code       56448 non-null  object 
 5   unit            56448 non-null  object 
 6   year            56448 non-null  int64  
 7   value           56448 non-null  float64
 8   value_bn        56448 non-null  float64
 9   region          56448 non-null  object 
dtypes: float64(2), int64(1), object(7)
memory usage: 4.3+ MB
None

--- GDP Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56448 entries, 0 to 56447
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_code    56448 non-null

,country_code,country,indicator_code,indicator,unit_code,unit,year,value,value_bn,region
0,AFG,Afghanistan,IMF_FFS_ECGFT,Fossil Fuel Subsidies - Total Implicit and Explicit,USD_K_2021,"US dollars, 2021 constant prices",2015,"921,795,165.40",0.92,South Asia
1,ALB,Albania,IMF_FFS_ECGFT,Fossil Fuel Subsidies - Total Implicit and Explicit,USD_K_2021,"US dollars, 2021 constant prices",2015,"363,617,142.90",0.36,Europe & Central Asia
2,DZA,Algeria,IMF_FFS_ECGFT,Fossil Fuel Subsidies - Total Implicit and Explicit,USD_K_2021,"US dollars, 2021 constant prices",2015,"39,137,999,790.00",39.14,Middle East & North Africa
3,AGO,Angola,IMF_FFS_ECGFT,Fossil Fuel Subsidies - Total Implicit and Explicit,USD_K_2021,"US dollars, 2021 constant prices",2015,"3,918,618,135.00",3.92,Sub-Saharan Africa
4,ARG,Argentina,IMF_FFS_ECGFT,Fossil Fuel Subsidies - Total Implicit and Explicit,USD_K_2021,"US dollars, 2021 constant prices",2015,"54,312,507,670.00",54.31,Latin America & Caribbean


In [25]:
# ============================================================
# SUBSIDY TRENDS: EXTERNALITIES, EXPLICIT, AND TOTAL
# ============================================================
import plotly.express as px

# Define the full set of indicators requested
all_indicators = [
    'Implicit Fossil Fuel Subsidies - Global Warming',
    'Implicit Fossil Fuel Subsidies - Local Air Pollution',
    'Implicit Fossil Fuel Subsidies - Congestion',
    'Implicit Fossil Fuel Subsidies - Road damage',
    'Implicit Fossil Fuel Subsidies - Accidents',
    'Implicit Fossil Fuel Subsidies - Foregone VAT',
    'Explicit Fossil Fuel Subsidies - Total',
    'Fossil Fuel Subsidies - Total Implicit and Explicit'
]

# Filter and group the USD data
full_trend = df_usd[
    df_usd['indicator'].isin(all_indicators)
].groupby(['year', 'indicator'])['value'].sum().reset_index()

# Clean up labels for the legend
full_trend['category'] = full_trend['indicator'].replace({
    'Implicit Fossil Fuel Subsidies - Global Warming': 'Implicit: Global Warming',
    'Implicit Fossil Fuel Subsidies - Local Air Pollution': 'Implicit: Air Pollution',
    'Implicit Fossil Fuel Subsidies - Congestion': 'Implicit: Congestion',
    'Implicit Fossil Fuel Subsidies - Road damage': 'Implicit: Road Damage',
    'Implicit Fossil Fuel Subsidies - Accidents': 'Implicit: Accidents',
    'Implicit Fossil Fuel Subsidies - Foregone VAT': 'Implicit: Foregone VAT',
    'Explicit Fossil Fuel Subsidies - Total': 'Explicit Subsidies',
    'Fossil Fuel Subsidies - Total Implicit and Explicit': 'TOTAL SUBSIDIES'
})

full_trend['value_bn'] = full_trend['value'] / 1e9

# Create the visual with markers and line labels
fig = px.line(
    full_trend, x='year', y='value_bn', color='category',
    title='<b>⚖️ Global Subsidy Breakdown: Externalities, Explicit, & Total (2015–2030)</b>',
    labels={'value_bn': 'USD Billions', 'year': 'Year', 'category': 'Subsidy Type'},
    markers=True,
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=px.colors.qualitative.Vivid
)

# Add direct labels to the end of each line
for i, d in enumerate(fig.data):
    fig.add_annotation(
        x=d.x[-1],
        y=d.y[-1],
        text=d.name,
        showarrow=False,
        xanchor='left',
        xshift=10,
        font=dict(size=10, color=d.line.color)
    )

fig.update_layout(
    height=700,
    width=1100,
    margin=dict(r=200), # Add space on the right for labels
    xaxis=dict(dtick=1),
    showlegend=False, # Legend is redundant with direct labels
    yaxis=dict(rangemode='tozero')
)
fig.show()

To be clear, explicit subsidies mean direct government price support (supply cost exceeds consumer price). This is what we normally think of as subsidies.

Implicit subsidies include local air pollution, climate change, congestion, road damage, accidents...the externalities that the fossil fuel industry fails to pay for.

As we can see from these projected trends, the IMF predicts that the fossil fuel industry will get away with more and more...even as direct government price supports remain constant. Dirty energy's worst sins are air pollution and global warming.

I hope to include per capita data as a follow-up.